# 00 — Set up the GMS stores

The chapter notebooks load **trained GMS stores** from `code/data/`
(`gms_banking_store`, `gms_policy_store`, `gms_regulatory_store`, and the
GEODE policy-RAG store). Those are build artifacts: **not committed and not
shipped in the package** (git-ignored, like all trained weights). A fresh
checkout has none of them, so most chapters fail with *"failed to load GMS
... store"* until this notebook has been run.

**What you need**

- The licensed **`knowlytix`** substrate: `pip install knowlytix` plus a
  developer license from https://knowlytix.ai/signup/ (key at
  `~/.knowlytix/license.key`). Without it no store can be built.
- `torch`.

**Two tiers**

1. **Tier 1 — CPU, no LLM.** The banking substrate store, its calibrated
   gate thresholds, the policy entity index and the regulatory guard. This
   is what most chapters (and the capstone tests) need. A few minutes.
2. **Tier 2 — GPU + Qwen.** The GEODE self-corrected policy-RAG store, used
   by the retrieval/answering chapters. Downloads Qwen3-4B-Instruct and
   expects a CUDA GPU.

Every stage is **idempotent** — skipped when its output already exists — so
you can re-run this notebook freely.

## 1. Bootstrap: locate `knowlytix` and this topic's code/ dir

In [ ]:
import importlib.util, os, sys

# Locate this topic's code/ dir, robust to the working directory.
_cwd = os.getcwd()
for REPO in (_cwd, os.path.join(os.path.dirname(_cwd), "code"), os.path.join(_cwd, "code")):
    if os.path.isdir(os.path.join(REPO, "agentlab")):
        break
else:
    REPO = os.environ.get("AGENTLAB_REPO", _cwd)
for p in (REPO, os.path.join(REPO, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

if importlib.util.find_spec('knowlytix') is None:
    raise ModuleNotFoundError(
        'knowlytix not found. Install it (`pip install knowlytix`, licensed) '
        'and put your key at ~/.knowlytix/license.key — see '
        'https://knowlytix.ai/signup/.')

import torch
print('torch:', torch.__version__, '| device:',
      'cuda' if torch.cuda.is_available() else 'cpu')
print('repo :', REPO)

## 1b. Preflight — dependencies and inputs


In [ ]:
# Preflight. The stages below load models and write stores, so anything wrong
# should surface here rather than several minutes in, and everything wrong is
# reported at once instead of one error per re-run.
import importlib.util
from pathlib import Path

_BOOK = 'beyond-prompt-and-pray'
_problems = []

# 1. Committed inputs this book's build chain reads.
for _rel in ['data/policies', 'data/bank_policies.md', 'data/banking_policy_full.md', 'data/eval_cases']:
    if not (Path(REPO) / _rel).exists():
        _problems.append(f"missing input {_rel!r} — expected in a complete clone")

# 2. What is already built. Reported before any hard failure, so this is still
#    informative without a licence. The lookup searches every book's data
#    directory: an artifact produced by one book and read by another is not
#    missing just because it lives with its producer.
try:
    from agentlab.artifacts import BOOK_MANIFESTS, missing_artifacts
    _declared = list(BOOK_MANIFESTS.get(_BOOK, ()))
    _absent = missing_artifacts(_BOOK)
    _have = [e for e in _declared if e not in _absent]
    print(f"artifacts for {_BOOK}: {len(_have)}/{len(_declared)} present")
    if _absent:
        print("  to be built by the stages below:")
        for _e in _absent:
            print("   -", _e)
    else:
        print("  nothing to build — every declared artifact is present")
except Exception as _e:                     # noqa: BLE001 - informational only
    print(f"(could not read the artifact manifest: {_e})")

# 3. Dependencies. Checked last so the report above is always shown.
for _mod, _why in [
    ("knowlytix", "the licensed GMS substrate — every stage below needs it"),
    ("torch", "tensor backend used to build and calibrate the stores"),
]:
    if importlib.util.find_spec(_mod) is None:
        _problems.append(f"missing package {_mod!r} — {_why}")

if _problems:
    print("\nPREFLIGHT FAILED — nothing has been built:\n")
    for _p in _problems:
        print("  -", _p)
    print(
        "\nknowlytix is free to install but gated by a runtime licence key:\n"
        "    pip install knowlytix\n"
        "    # then place your key at ~/.knowlytix/license.key\n"
        "    # sign up at https://knowlytix.ai/signup/\n"
        "Missing inputs usually mean a partial clone or a deleted data/ directory."
    )
    raise RuntimeError("preflight failed; see the list above")

print("\npreflight ok — inputs present, dependencies importable")


## 2. Idempotent stage runner

In [ ]:
import subprocess, sys, os

def stage(title, script, args=(), produces=()):
    """Run scripts/<script> unless every path in `produces` already exists.
    Streams the script's output; raises on non-zero exit."""
    produces = list(produces)
    if produces and all(os.path.exists(os.path.join(REPO, p)) for p in produces):
        print(f'\u2713 {title}: already built \u2014 skipping')
        return
    cmd = [sys.executable, os.path.join(REPO, 'scripts', script), *map(str, args)]
    print(f'\u25b6 {title}: python scripts/{script} ' + ' '.join(map(str, args)))
    r = subprocess.run(cmd, cwd=REPO, env=os.environ.copy())
    if r.returncode:
        raise RuntimeError(f'{script} failed (exit {r.returncode})')
    print(f'\u2713 {title}: done')

## 3. Tier 1 — CPU stores (required)

No LLM and no GPU. The banking store is the Chapter-16 substrate (ENM,
tension, plausibility gate); calibrating it writes the gate thresholds the
capstone reads. The policy and regulatory stores back `search_policy` and
`flag_regulatory`.

In [ ]:
stage('Banking substrate store', 'retrain_gms_banking.py',
      produces=['data/gms_banking_store/model.pt'])

stage('Calibrate plausibility/contradiction thresholds', 'calibrate_gms_thresholds.py',
      produces=['data/gms_banking_store/calibration.json'])

stage('Policy entity index (Graph-RAG)', 'build_policy_rag_store.py',
      produces=['data/gms_policy_store/model.pt'])

stage('Regulatory guard store', 'build_regulatory_guard_store.py',
      produces=['data/gms_regulatory_store/model.pt'])

stage('Calibrate regulatory guard thresholds', 'calibrate_regulatory_guard.py',
      produces=['data/gms_regulatory_store/calibration.json'])

## 4. Verify the Tier-1 stores load

In [ ]:
import json, os

for name in ('gms_banking_store', 'gms_policy_store', 'gms_regulatory_store'):
    p = os.path.join(REPO, 'data', name)
    print(f"{name:24} {'OK ' if os.path.isdir(p) else 'MISSING'}")

cal = os.path.join(REPO, 'data', 'gms_banking_store', 'calibration.json')
if os.path.isfile(cal):
    theta = json.load(open(cal))['plausibility_gate']['threshold']
    print('calibrated plausibility threshold:', theta)

## 5. Tier 2 — GEODE policy-RAG store (optional; GPU + Qwen)

The retrieval/answering chapters bind queries through a GEODE
self-corrected policy graph. Building it runs Qwen3-4B-Instruct and
**expects a CUDA GPU**; it also downloads the model (~6 GB).

**Two store names, one build.** Consumers are split:
`gms_policy_store_cap` is read by `agentlab.capstone.policy_rag` and the
Ch15 capstone; `gms_policy_store_geode` is read by the claim-route /
entity-link extractors and by beyond-ship-and-pray's chapters. They are the
*same* artifact — `build_geode_rag_store.py` always trains `loss_mode="cap"`
and only the `--store-path` differs (its default, `_geode`, is not what the
capstone reads). So build once to `_cap` and mirror it to `_geode` rather
than paying for a second GEODE+Qwen run. If the two ever need to diverge,
run the script twice with explicit `--store-path`.

**Value-polarity verifier.** `build_policy_value_polarity.py` builds the
fused stance-verification gate used by `search_policy` (Ch16 Supplement 3)
and the governed-retrieval store (Ch13). It writes two artifacts into
`gms_policy_store_cap`: a fine-tuned u-space polarity encoder
(`value_polarity_encoder/`) and calibrated 3-class tension cuts
(`value_polarity_calibration.json`). This stage runs immediately after the
GEODE store build so the mirror step picks up both artifacts.

**Chapter 13 governed store.** `build_governed_store.py` builds
`gms_governed_store` — the GEODE store over the governed corpus plus the
governance layer (sensitivity map + retrieval contracts, value-polarity
verifier, calibrated RAG and disclosure gates) — by chaining the existing
builders in dependency order. `--scenarios` also produces the
`governed_scenarios` / `polarity_gate_comparison` artifacts the Chapter 13
notebook embeds.

Set `RUN_TIER2 = True` to build.


In [ ]:
import shutil

RUN_TIER2 = torch.cuda.is_available()  # auto-detect GPU; set False to skip Tier 2

if RUN_TIER2:
    stage('GEODE policy-RAG store (loads Qwen)', 'build_geode_rag_store.py',
          args=['--store-path', 'data/gms_policy_store_cap'],
          produces=['data/gms_policy_store_cap/model.pt'])

    # Value-polarity verifier (Ch13/16): fine-tunes a u-space polarity encoder
    # and calibrates the 3-class tension cuts, both written into gms_policy_store_cap.
    # Must run after the GEODE store so the smoke-test step can load it.
    stage('Value-polarity verifier (Ch13/16; loads Qwen)', 'build_policy_value_polarity.py',
          args=['data/gms_policy_store_cap'],
          produces=['data/gms_policy_store_cap/value_polarity_encoder/meta.json',
                    'data/gms_policy_store_cap/value_polarity_calibration.json'])

    # Mirror to the name the extractors / beyond-ship-and-pray expect.
    _cap = os.path.join(REPO, 'data', 'gms_policy_store_cap')
    _geode = os.path.join(REPO, 'data', 'gms_policy_store_geode')
    if os.path.isdir(_cap) and not os.path.isdir(_geode):
        shutil.copytree(_cap, _geode)
        print('✓ mirrored gms_policy_store_cap -> gms_policy_store_geode')

    # Seed train/valid split for the complaint classifier (from eval_cases/cases.json).
    stage('Complaint training seeds', 'build_complaint_training_seeds.py',
          produces=['data/training/complaint_classification/train.jsonl'])

    # DoE-augmented complaint corpus (Ch16 Supplement 9): label-preserving paraphrases
    # of the training seeds across presentation factors; produces train_doe.jsonl and
    # test_doe.jsonl used by 16_supplement_9_doe_data_enrichment.ipynb.
    stage('Complaint DoE corpus (Ch16 Supplement 9; loads Qwen)', 'augment_complaint_training_doe.py',
          produces=['data/training/complaint_classification/train_doe.jsonl',
                    'data/training/complaint_classification/test_doe.jsonl'])

    # Complaint classifier (Ch15/Ch16 capstone): Qwen LoRA head trained on the
    # DoE-augmented complaint corpus.
    stage('Complaint classifier (Ch15/Ch16; loads Qwen)', 'train_complaint_classifier_qwen.py',
          produces=['data/complaint_classifier_qwen/adapter_config.json'])

    # DoE-augmented {none, prompt_injection, prohibited_advice} corpus; seeds
    # governance_exemplars.json + hard-negatives + a sample of complaint/inquiry
    # messages, rephrased across clarity levels by Qwen. Consumes train.jsonl
    # (complaint seeds) so must run after the complaint-seeds stage.
    stage('Injection detector corpus (Ch16; loads Qwen)', 'build_injection_corpus.py',
          produces=['data/training/injection_doe.jsonl'])

    # Injection/prohibited-advice classifier (Ch16 Policy gate): Qwen LoRA
    # SEQ_CLS head trained on the corpus above. Replaces the brittle zero-shot
    # fallback that over-fires on benign "skip the formalities" phrasings.
    stage('Injection classifier (Ch16; loads Qwen)', 'train_injection_classifier.py',
          produces=['data/injection_classifier_lora/labels.json'])

    # MeMo SFT corpus for the draft_response LoRA (complaint-shaped
    # policy-grounded reply pairs validated against the goldens). Needs the
    # policy GMS store (already built above) to generate grounded numbers.
    stage('Draft-response corpus (Ch15/16; loads Qwen)', 'build_draft_response_corpus.py',
          produces=['data/training/bank_policy/draft_response_memo_v3.jsonl'])

    # Draft-response LoRA adapter (Ch15/16 capstone): Qwen fine-tuned on the
    # MeMo corpus above. Missing adapter causes every complaint that reaches
    # draft_response to fail, which did_escalate() counts as escalation.
    stage('Draft-response LoRA (Ch15/16; loads Qwen)', 'train_draft_response_lora.py',
          produces=['data/draft_response_lm_qwen/adapter_config.json'])

    # Regulatory DoE corpus (Ch16 Supplement 4): label-preserving paraphrases of
    # the eval seeds across clarity/style factors, used to train the rank-1 adapter
    # in the next stage. Reads eval_cases/cases.json (committed); loads Qwen.
    stage('Regulatory DoE corpus (Ch16; loads Qwen)', 'augment_regulatory_doe.py',
          produces=['data/training/regulatory_doe_augmented.jsonl'])

    # Regulatory cap store (Ch16 Supplement 4): fine-tuned embedding + per-flag
    # spherical caps for the ManifoldFlagScorer geometric reader. Inherits device
    # from gms_regulatory_store (Tier 1). Missing artifact degrades gracefully
    # (scorer returns None), but the Supplement 4 notebook shows the cap-reader
    # demo only when the artifact is present.
    stage('Regulatory cap store (Ch16; SFT)', 'build_regulatory_cap_store.py',
          produces=['data/gms_regulatory_cap/calibration.json'])

    # Chapter 13 governed-retrieval store: the GEODE store built from the
    # governed corpus, plus the governance layer (sensitivity map + contracts,
    # value-polarity verifier, calibrated RAG and disclosure gates). One
    # orchestrator chains the existing builders in dependency order; --scenarios
    # also writes the governed_scenarios / polarity_gate_comparison artifacts the
    # Ch13 notebook embeds.
    stage('Governed-retrieval store (Ch13; loads Qwen)', 'build_governed_store.py',
          args=['--scenarios'],
          produces=['data/gms_governed_store/model.pt',
                    'data/gms_governed_store/rag_gate_calibration.json',
                    'data/gms_governed_store/disclosure_gate_calibration.json'])

    # Graph-truth retrieval benchmark (Ch9): grades search_policy against the
    # policy graph ground truth and produces the GMS-vs-dense comparison the
    # Ch9 notebook embeds.
    stage('Graph-truth retrieval benchmark (Ch9)', 'build_capstone_retrieval.py',
          produces=['data/capstone_retrieval.json'])

    print('\nTier 2 complete.')
else:
    print('RUN_TIER2 is False — Tier-1 stores only. The retrieval/answering '
          'chapters and the end-to-end capstone tests need Tier 2.')

## Done

The stores live under `code/data/` (git-ignored, never packaged). Open the
chapter notebooks — they load these directly. Re-run this notebook any time;
existing stores are skipped.